# 04 — Compare and report

**Depends on:** notebooks 00–03. **Produces:** `leaderboard.csv`, `report.md`,
`figures/`.

## What this stage is for

Picking a winner, checking that the win is real, and writing it down so that
somebody who was not here can read the decision and disagree with it on the
merits.

## How to pick a winner without fooling yourself

Three habits:

1. **Fix the operating point before looking at the numbers.** Decide k from the
   review capacity, not from where the curves look best. Choosing k after seeing
   the curves is choosing the metric that flatters your favourite model.
2. **Look at the spread, not the maximum.** If the top five are within noise, you
   do not have a winner, you have five candidates — and then the tiebreak should
   be cost, interpretability and maintenance, not the fourth decimal place.
3. **Bootstrap the interval.** With a few hundred positives, `precision@500` has
   a confidence interval wide enough to swallow most of the differences on the
   leaderboard. The cell below computes it. Read it before you commit to anything.

## What the report is for

Three weeks from now nobody will remember why this model was chosen. The report
is the artefact that answers it. It states the operating point, the winner, the
margin, the confidence interval, and — importantly — what was *not* tested.

In [ ]:
import json
import os
import pathlib
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Every notebook in this track reads and writes here. Nothing else is shared:
# no imports between notebooks, no global state, no kernel that has to stay
# alive. That is what "standalone" means -- you can run notebook 03 tomorrow on
# a different machine as long as this directory came with you.
RUN_DIR = pathlib.Path(os.environ.get("TS05_RUN_DIR", "./_run")).resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)

SEED = 20260601
rng = np.random.default_rng(SEED)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

print("run directory:", RUN_DIR)
print("contents     :", sorted(p.name for p in RUN_DIR.iterdir()) or "(empty)")

In [ ]:
manifest = json.loads((RUN_DIR / "manifest.json").read_text())
assert "03" in manifest["stages_complete"], "Run 03-train-and-score.ipynb first."

scores = pd.read_parquet(manifest["paths"]["scores"])
metrics = pd.read_csv(manifest["paths"]["metrics"])
selection = json.loads((RUN_DIR / "selected_features.json").read_text())
quality = json.loads((RUN_DIR / "quality_report.json").read_text())

model_names = manifest["trained"]["models"]
base_rate = manifest["trained"]["base_rate"]

print("models   :", model_names)
print("base rate:", f"{base_rate:.4%}")
print("k values :", manifest["trained"]["k_values"])

## Fix the operating point

Set this from the review capacity. Say the fraud team can work 500 customers a
week — then k=500 and the metric is `precision_at_k`. Write the reasoning down;
it is the most contestable decision in the whole pipeline and it should be
contestable in the open.

In [ ]:
OPERATING_K = 160
PRIMARY_METRIC = "precision_at_k"
RATIONALE = (
    "The fraud review team works roughly 500 customers per week, so the queue "
    "length is 500 and the number that matters is what fraction of that queue "
    "is real. Recall and lift are reported alongside but do not decide the "
    "winner."
)

available = sorted(metrics["k"].unique())
OPERATING_K = min(available, key=lambda k: abs(k - OPERATING_K))
print(f"operating k    : {OPERATING_K} (nearest available to the target)")
print(f"primary metric : {PRIMARY_METRIC}")
print()
print(RATIONALE)

## The leaderboard

In [ ]:
leaderboard = (
    metrics[metrics["k"] == OPERATING_K]
    .sort_values(PRIMARY_METRIC, ascending=False)
    .reset_index(drop=True)
)
leaderboard["rank"] = leaderboard.index + 1

columns = ["rank", "model", "hits", "precision_at_k", "recall_at_k", "lift_at_k",
           "roc_auc", "average_precision"]
display(leaderboard[columns].round(4))

champion = leaderboard.iloc[0]["model"]
print()
print("provisional champion:", champion)

## Is the margin real?

Bootstrap the evaluation population and recompute `precision@k`. The interval
this produces is usually much wider than people expect, and it is the difference
between "model A is better" and "model A came first this time".

In [ ]:
N_BOOTSTRAP = 300
labels = scores["label"].to_numpy()
n = len(labels)

boot = {name: [] for name in model_names}
for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n, size=n)
    sample_labels = labels[idx]
    for name in model_names:
        values = scores[name].to_numpy()[idx]
        order = np.argsort(-values)[:OPERATING_K]
        boot[name].append(sample_labels[order].mean())

intervals = pd.DataFrame(
    [
        {
            "model": name,
            "mean": float(np.mean(values)),
            "p05": float(np.quantile(values, 0.05)),
            "p95": float(np.quantile(values, 0.95)),
        }
        for name, values in boot.items()
    ]
).sort_values("mean", ascending=False).reset_index(drop=True)
intervals["width"] = intervals["p95"] - intervals["p05"]

display(intervals.round(4))

best_low = intervals.iloc[0]["p05"]
overlapping = intervals[intervals["p95"] >= best_low]["model"].tolist()
print()
print(f"{len(overlapping)} model(s) whose 90% interval overlaps the leader's:")
print(overlapping)
if len(overlapping) > 1:
    print()
    print("The lead is NOT statistically clean. Break the tie on cost, "
          "interpretability and maintenance burden, and say so in the report.")

In [ ]:
import matplotlib.pyplot as plt

FIGURE_DIR = RUN_DIR / "figures"
FIGURE_DIR.mkdir(exist_ok=True)

figure, axis = plt.subplots(figsize=(8, 0.5 * len(intervals) + 2))
order = intervals.iloc[::-1]
axis.barh(order["model"], order["mean"], xerr=[
    order["mean"] - order["p05"], order["p95"] - order["mean"]
], capsize=4)
axis.axvline(base_rate, linestyle="--", color="grey", label="base rate")
axis.set_title(f"precision@{OPERATING_K} with 90% bootstrap interval")
axis.set_xlabel("precision")
axis.legend()
plt.tight_layout()
figure.savefig(FIGURE_DIR / f"precision_at_{OPERATING_K}.png", dpi=140, bbox_inches="tight")
plt.show()

## The whole curve, not one point

A model that wins at k=500 and loses at k=100 is a model whose ranking is good in
the middle and poor at the top. If the queue length is ever going to change, the
shape of the curve matters more than the single point.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for name in model_names:
    subset = metrics[metrics["model"] == name].sort_values("k")
    axes[0].plot(subset["k"], subset["precision_at_k"], marker="o", label=name)
    axes[1].plot(subset["recall_at_k"], subset["precision_at_k"], marker="o", label=name)

axes[0].axvline(OPERATING_K, linestyle="--", color="black", linewidth=0.9)
axes[0].axhline(base_rate, linestyle="--", color="grey")
axes[0].set_xscale("log")
axes[0].set_xlabel("k")
axes[0].set_ylabel("precision")
axes[0].set_title("precision@k (operating point marked)")
axes[1].set_xlabel("recall")
axes[1].set_ylabel("precision")
axes[1].set_title("precision against recall")
axes[0].legend(fontsize=7)
plt.tight_layout()
figure.savefig(FIGURE_DIR / "precision_recall_curves.png", dpi=140, bbox_inches="tight")
plt.show()

## What the champion is actually reacting to

No permutation importance and no SHAP — those need a supervised target. Instead:
take the top-k customers by score and compare their feature distribution against
the population. The features where they differ most are what the model is keying
on, and they should be recognisable as fraud-like. If the top of the queue is
distinguished by something that makes no operational sense, that is the finding
to chase.

In [ ]:
training = pd.read_parquet(manifest["paths"]["training_nonfraud"])
testing = pd.read_parquet(manifest["paths"]["testing_nonfraud"])
fraud = pd.read_parquet(manifest["paths"]["fraud"])
population = pd.concat([testing, fraud], ignore_index=True)

features = manifest["trained"]["features"]
flagged_ids = set(scores.nlargest(OPERATING_K, champion)["customer_id"])
flagged = population[population["customer_id"].isin(flagged_ids)]

rows = []
for name in features:
    a = population[name].dropna().astype(float)
    b = flagged[name].dropna().astype(float)
    if len(b) < 10 or a.std(ddof=1) == 0:
        continue
    rows.append(
        {
            "feature": name,
            "population_mean": a.mean(),
            "flagged_mean": b.mean(),
            "std_diff": (b.mean() - a.mean()) / a.std(ddof=1),
        }
    )

drivers = pd.DataFrame(rows)
drivers["abs"] = drivers["std_diff"].abs()
drivers = drivers.sort_values("abs", ascending=False)

print(f"what distinguishes the top-{OPERATING_K} flagged by {champion}:")
display(drivers.head(15).round(3))

figure, axis = plt.subplots(figsize=(9, 5))
top = drivers.head(15).iloc[::-1]
axis.barh(top["feature"], top["std_diff"])
axis.axvline(0, color="black", linewidth=0.8)
axis.set_title(f"{champion}: flagged-vs-population mean difference (std units)")
axis.tick_params(axis="y", labelsize=8)
plt.tight_layout()
figure.savefig(FIGURE_DIR / "champion_drivers.png", dpi=140, bbox_inches="tight")
plt.show()

## Where the misses are

The fraud that scored *low* is more informative than the fraud that scored high.
If the misses share a pattern, that pattern is a feature the library does not
have — and that is the most actionable output of this whole track.

In [ ]:
fraud_scores = scores[scores["label"] == 1].copy()
fraud_scores["percentile"] = fraud_scores[champion].rank(pct=True)

print("fraud score percentile distribution:")
print(fraud_scores["percentile"].describe().round(3).to_string())

missed = fraud_scores.nsmallest(max(len(fraud_scores) // 5, 5), champion)
missed_rows = population[population["customer_id"].isin(set(missed["customer_id"]))]
caught_ids = set(scores.nlargest(OPERATING_K, champion)["customer_id"])
caught_rows = population[
    population["customer_id"].isin(caught_ids & set(fraud_scores["customer_id"]))
]

print()
print(f"comparing the {len(missed_rows)} lowest-scoring fraud against "
      f"the {len(caught_rows)} caught:")
comparison = pd.DataFrame(
    {
        "missed_mean": missed_rows[features].mean(),
        "caught_mean": caught_rows[features].mean(),
    }
)
comparison["difference"] = comparison["caught_mean"] - comparison["missed_mean"]
display(comparison.reindex(comparison["difference"].abs().sort_values(
    ascending=False).index).head(15).round(3))

## Write the report

In [ ]:
import datetime as dt

leader = intervals.iloc[0]
runner_up = intervals.iloc[1] if len(intervals) > 1 else None

lines = [
    "# Trust Score — standalone track results",
    "",
    f"Generated {dt.datetime.now().isoformat(timespec='minutes')} from `{RUN_DIR}`.",
    "",
    "## What was done",
    "",
    f"Five notebooks, run in order, on "
    f"{'synthetic' if manifest['synthetic'] else 'real'} data: "
    f"{manifest['n_customers']:,} customers, "
    f"{manifest['n_fraud_customers']:,} of them confirmed fraud.",
    "",
    f"Feature selection kept **{len(features)}** of "
    f"{len(selection['selected'][manifest['trained']['top_n']])} ranked features "
    f"(the top-{manifest['trained']['top_n']} list). "
    f"Excluded as label leakage: {selection['excluded_as_leakage'] or 'none'}.",
    "",
    f"**{len(model_names)}** models were fitted on the decontaminated non-fraud "
    f"population and scored against a mixed evaluation population with a base "
    f"rate of {base_rate:.4%}.",
    "",
    "## Operating point",
    "",
    RATIONALE,
    "",
    f"- queue length (k): **{OPERATING_K}**",
    f"- primary metric: **{PRIMARY_METRIC}**",
    "",
    "## Result",
    "",
    f"**{leader['model']}** leads with precision@{OPERATING_K} = "
    f"**{leader['mean']:.4f}** "
    f"(90% bootstrap interval {leader['p05']:.4f}–{leader['p95']:.4f}), "
    f"a lift of "
    f"{leaderboard.iloc[0]['lift_at_k']:.1f}x over the base rate.",
    "",
]

if runner_up is not None:
    clean = runner_up["p95"] < leader["p05"]
    lines += [
        f"The runner-up is **{runner_up['model']}** at {runner_up['mean']:.4f} "
        f"({runner_up['p05']:.4f}–{runner_up['p95']:.4f}). The intervals "
        f"{'do not overlap, so the lead is statistically clean' if clean else 'overlap, so the lead is NOT statistically clean'}.",
        "",
    ]

if len(overlapping) > 1:
    lines += [
        f"In total {len(overlapping)} models are statistically "
        f"indistinguishable from the leader: {', '.join(overlapping)}. "
        "The choice between them should be made on cost, interpretability and "
        "maintenance burden rather than on this metric.",
        "",
    ]

lines += [
    "## Leaderboard",
    "",
    leaderboard[columns].round(4).to_markdown(index=False),
    "",
    "## What the champion keys on",
    "",
    "The strongest differences between the flagged queue and the population, "
    "in standard deviations:",
    "",
    drivers.head(10)[["feature", "population_mean", "flagged_mean", "std_diff"]]
    .round(3)
    .to_markdown(index=False),
    "",
    "## Data quality at the time of this run",
    "",
    f"Worst severity: **{quality['severity']}**.",
    "",
]
for entry in quality["checks"]:
    if entry["severity"] != "info":
        lines.append(f"- **{entry['severity']}** — {entry['name']}: {entry['message']}")
lines += [
    "",
    "## What was NOT tested",
    "",
    "Stated explicitly, because the omissions are where the next surprise comes from:",
    "",
    "- Only one feature-list length was trained. Length is a hyperparameter and "
    "was not swept here; the package track sweeps it.",
    "- Default hyperparameters throughout. No grid search.",
    "- One preprocessing recipe (domain imputation plus a robust scaler). "
    "The alternatives were not compared.",
    "- One evaluation scenario. The package defines several and requires a "
    "configuration to win on all of them.",
    "- No temporal validation. Every split here is random, so nothing here "
    "measures whether the model degrades over time.",
    "- No drift monitoring. See `notebooks/3.0-modeling/3.4-drift.ipynb`.",
    "",
    "## Next steps",
    "",
    "1. Re-run this track against the real feature matrix by setting "
    "`USE_SYNTHETIC = False` in notebook 00.",
    "2. Move to the package track for the full sweep — it covers all eleven "
    "models, the hyperparameter grids, every preprocessing recipe and all "
    "scenarios, and it is resumable.",
    "3. Add temporal validation before anything ships.",
    "",
]

report_text = "\n".join(lines)
(RUN_DIR / "report.md").write_text(report_text)
leaderboard.to_csv(RUN_DIR / "leaderboard.csv", index=False)
intervals.to_csv(RUN_DIR / "bootstrap_intervals.csv", index=False)

manifest["stages_complete"] = sorted(set(manifest["stages_complete"]) | {"04"})
manifest["paths"].update(
    {
        "leaderboard": str(RUN_DIR / "leaderboard.csv"),
        "report": str(RUN_DIR / "report.md"),
        "figures": str(FIGURE_DIR),
    }
)
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2))

print(report_text)

## Done

Everything is in the run directory: the frames, the schema, the quality report,
the ranked features, the fitted bundles, the scores, the metrics, the figures and
`report.md`. The directory is self-contained — hand it to somebody and they can
read the whole decision without this kernel.

**Where to go next.** The package track. This track proves the pipeline works and
teaches what each step does; the package track is what runs on real data at real
scale, with the sweep, the resumability and the tests. Start at
`notebooks/1.0-exploration/1.1-schema-discovery.ipynb`.